# <center><u> **COMP9444 Group Assignment - (Nom Nom Nom)** </u></center>
### <center>"Question Answering on Movie Knowledge Graphs"</center>
###### <center>Lachlan Lloyd (z5764009), Daniel Guo (z5590943), Adit Mahidadia (z5588982), Brandon Zhang (z5592981)</center>
                                                                                                    

-----------------------------------

## **Introduction**

Knowledge graphs (KGs) and natural language processing (NLP) are often regarded as quite specialised topics within machine learning, yet the systems built upon them are used extensively across many fields. Whenever a user searches the web, asks Siri or Alexa a question, receives movie recommendations on Netflix or is shown a recommended add, they are, by extension, interacting with a knowledge graphs and the question answering models built on top of them. This project examines how these systems operate by developing and evaluating a model that answers natural language questions using a KG. The work progresses from understanding the structure of the KG, to training a model capable of traversing it, to evaluating the model's performance on unseen questions.

### **Problem Statement**

A knowledge graph represents facts as triples of the form (*head entity*, *relation*, *tail entity*), for example (*Chopper*, *directed_by*, *Andrew Dominik*). Knowledge Graph Question Answering (KGQA) is the task of mapping a natural language question, such as "who directed Chopper?", to a correct entity in the graph. Reliable KGQA has significant practical value in areas like recommender systems, pharmaceutical drug and hazard relationships or basic voice assistants like Alexa and Siri.


This project trains a model on "MetaQA", a movie based KG with over 400,000 question-answer pairs. It is capable of answering unseen test questions about cast, directors, genres and release years. Typical KGQA tasks have two main problems or limitations: **Graph Completness and Multi-Hop Reasoning**

- Multi-hop reasoning: many questions require traversing multiple edges of the graph, including inverse relations (e.g. "which genres are the movies directed by the director of X?"), and model accuracy will typically degrades as the number of hops increases
- Graph Completness: many real-world KGs contain missing links so the correct answer may not be reachable through a direct path. Methods that restrict possible answers to a fixed "neighbourhood" around the topic entity can end up excludeing the true answer entirely


To address these challenges, we explore existing propositions and develop implementations for them. We train and evaluate these implementations as our task demands, but also experiment with extensions on the existing solutions in an attempt to generate more accurate responses to factual questions about movies. The 3 selected baseline models we will develop, train and evaluate are the Variational Reasoning Network (VRN) [1], EmbedKGQA [2] and PullNet [3].


[1] Y. Zhang, H. Dai, Z. Kozareva, A. J. Smola, and L. Song, “Variational Reasoning for Question Answering with Knowledge Graph,” Nov. 27, 2017, arXiv: arXiv:1709.04071. doi: 10.48550/arXiv.1709.04071.

[2] A. Saxena, A. Tripathi, and P. Talukdar, “Improving Multi-hop Question Answering over Knowledge Graphs using Knowledge Base Embeddings,” in Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics, Online: Association for Computational Linguistics, 2020, pp. 4498–4507. doi: 10.18653/v1/2020.acl-main.412.

[3] H. Sun, T. Bedrax-Weiss, and W. W. Cohen, “PullNet: Open Domain Question Answering with Iterative Retrieval on Knowledge Bases and Text,” Apr. 21, 2019, arXiv: arXiv:1904.09537. doi: 10.48550/arXiv.1904.09537.

## **Data Sources**
We use the MoviE Text Audio QA, or the [MetaQA dataset](https://github.com/yuyuz/MetaQA). It contains three different components: Vanilla, NTM and Audio. Each component contains data split into three different hops, 1 hop, 2 hop and 3 hop, or the number of edges that needs to be traversed in the KG from the question to an answer. Vanilla is text data, with 1 hop data is derived from the wiki_entities branch of the WikiMovies dataset, with questions containing ambiguous entities removed. Vanilla 2 hop and 3 hop data are generated from this same knowledge base. NTM is Vanilla text that has undergone translation from English to French then back to English using a neural translation model. Audio is Vanilla text with all questions read out loud by Google's text to speech API.

Vanilla and NTM store text data as question answer pairs separated by a `\t`, with one per line. Questions are written as a knowledge triple, in the schema `subject|relation|object`. Any entities within these questions are marked by a pair of square brackets, like `question text [Entity] more question text`. Answers are written as `answer1|answer2|..|answerN` or just `answer` if only one answer is correct. Audio stores audio data as mp3 files, with one file for the question per question answer pair. Each file name follows the 0 indexed line index of the questions in Vanilla.

### **Exploratory Analysis of Data**
The question answer pairs are partitioned into test, dev and train splits. All 3 components share the same split of question answer pairs.
|       |  1 hop |   2 hop |   3 hop |
|-------|-------:|--------:|--------:|
| Train | 96,106 | 118,980 | 114,196 |
| Dev   |  9,992 |  14,872 |  14,274 |
| Test  |  9,947 |  14,872 |  14,274 |

1 hop data is made up of 13 different types of questions, 2 hop has 21 question types and 3 hop has 15 question types. Each type of question has a further 10 templates, which are different ways to word a question but have the same meanining.
- 1 hop questions:
    - Actor -> Movie
    - Director -> Movie
    - Writer -> Movie
    - Tag -> Movie
    - Movie -> Actor, Director, Writer, Genre, Language, Year, Tags, IMDb Rating, IMDb Votes
- 2 hop questions:
    - Movie -> Actor, Director, Writer -> Movie
    - Actor -> Movie -> Actor, Director, Genre, Language, Writer, Year
    - Director -> Movie -> Actor, Director, Genre, Language, Writer, Year
    - Writer -> Movie -> Actor, Director, Genre, Language, Writer, Year
- 3 hop questions:
    - Movie -> Actor -> Movie -> Director, Genre, Language, Writer, Year
    - Movie -> Director -> Movie -> Actor, Genre, Language, Writer, Year
    - Movie -> Writer -> Movie -> Actor, Director, Genre, Language, Year

The main challenge of this data is its enormity (and early development of models hinting that the accuracy of any model we development is proportional to the number of samples we test against) and a lack of compute (none of our laptops are powerful enough so we did everything using Kaggle or Amazon's EC2), we have limited our focus to just the Vanilla component. However, the simple format of the data and extremely easy way to evaluate it (simply checking if the model will produce an answer that's in )

To streamline pre processing, all models we developed use a common `base.py` file, containing the classes `Example` (a question answer pair) and `KnowledgeGraph` (the KG). It also contains two helpers `parse_question` (turn a question answer pair into an `Example`) and `load_qa` (from MetaQA, load the question answer pairs for the given hop in 1, 2 or 3 hop and split in train, test or dev).

All imports and constants.

In [ ]:
from __future__ import annotations
from dataclass import dataclass
from collections import defaultdict
from pathlib import Path
from typing import Final
import re

INVERSE: Final = "inverse_"
ENTITY_REGEX: Final = re.compile(r"\[([^\]]+)\]")
WORD_REGEX: Final = re.compile(r"[a-z0-9]+")

The `Example` dataclass to represent each question answer pair, with preprocessing to extract any topic entities from the question, and extracting all words from the question string to get the template being used.

In [ ]:
@dataclass(frozen=True)
class Example:
    question: str
    topic: str
    template: str
    answers: frozenset[str]

The `parse_question` helper function to convert question answer pairs into `Example`s.

In [ ]:
def parse_question(question: str, answers: str) -> Example:
    match = ENTITY_REGEX.search(question)
    if not match:
        raise ValueError(f"Missing a [topic entity] in question: {question}")
    topic = match.group(1).strip()
    template = ENTITY_REGEX.sub(" ENTITY ", question, count=1).lower()
    template = " ".join(WORD_REGEX.findall(template))
    answer_set = frozenset(answer.strip() for answer in answers.split("|"))
    if not answer_set:
        raise ValueError(f"Question has no answers: {question}")
    return Example(question, topic, template, answer_set)

The `load_qa` helper function to convert a text file of question answer pairs, loaded from MetaQA's Vanilla component given the specific 1 or 2 or 3 hop and test or train or dev split.

In [ ]:
def load_qa(root: Path, hop: int, split: str) -> list[Example]:
    path = root / f"{hop}-hop" / "vanilla" / f"qa_{split}.txt"
    if not path.is_file():
        raise FileNotFoundError(f"QA file could not be found at path={path}")
    result: list[Example] = []
    with path.open(encoding="utf-8-sig") as file:
        for line_number, data in enumerate(file, 1):
            data = data.strip()
            if not data:
                continue
            try:
                question, answers = data.split("\t", maxsplit=1)
            except ValueError as e:
                raise ValueError(r"Expected QUESTION\tANSWERS at" + f"{path}:{line_number}") from e
            result.append(parse_question(question, answers))
    return result

The `KnowledgeGraph` class representing the KG of MetaQA, including some helper functions for models to use it.

In [ ]:
class KnowledgeGraph:
    def __init__(self, kb_path: Path) -> None:
        """
        Initialise the Knowledge Graph from the given file
        """
        self.edges: dict[str, dict[str, set[str]]] = defaultdict(lambda: defaultdict(set))
        self.triples = 0

        with kb_path.open(encoding="utf-8-sig") as file:
            for line_number, data in enumerate(file, 1):
                data = data.strip()
                if not data:
                    continue

                triple = data.split("|")
                if len(triple) != 3:
                    raise ValueError(f"Expected subject|relation|target at {kb_path}:{line_number}")

                subject, relation, target = (part.strip() for part in triple)

                self.edges[subject][relation].add(target)
                self.edges[target][INVERSE + relation].add(subject)
                self.triples += 1

    def follow(self, topic_entity: str, relation_path: tuple[str, ...]) -> set[str]:
        """
        Gets all final entities reachable by following the given relation path starting from the topic entity
        """
        nodes: set[str] = {topic_entity}
        for relation in relation_path:
            next_nodes = set[str]()
            for node in nodes:
                targets = self.edges.get(node, {}).get(relation, ())
                for target in targets:
                    next_nodes.add(target)
            nodes = next_nodes
            if not nodes:
                break
        return nodes

    def all_paths(self, topic_entity: str, hops: int) -> dict[tuple[str, ...], set[str]]:
        """
        Gets all combinations of relations and final entities found by following them, within hops distance from topic entity
        """
        states: dict[tuple[str, ...], set[str]] = {(): {topic_entity}}
        for _ in range(hops):
            next_states: dict[tuple[str, ...], set[str]] = defaultdict(set)
            for relation_path, nodes in states.items():
                for node in nodes:
                    for relation, targets in self.edges.get(node, {}).items():
                        next_states[relation_path + (relation,)].update(targets)
            states = next_states
            if not states:
                break
        return states

    def get_all_entities(self) -> set[str]:
        """
        Get all entities in the Knowledge Graph
        """
        return set(self.edges.keys())

    def get_all_relations(self, include_inverse: bool = True) -> set[str]:
        """
        Get all relations in the Knowledge Graph
        """
        relations = {
            relation
            for node_relations in self.edges.values()
            for relation in node_relations
        }

        if not include_inverse:
            relations = {
                relation for relation in relations
                if not relation.startswith(INVERSE)
            }

        return relations

    def get_all_triples(self, include_inverse: bool = True) -> set[tuple[str,str,str]]:
        """
        Get all facts, or (subject, relation, object) triples in the Knowledge Graph
        """
        return {
        (source, relation, target)
        for source, relations in self.edges.items()
        for relation, targets in relations.items()
        if include_inverse or not relation.startswith(INVERSE)
        for target in targets
        }

    def has_triple(self, triple: tuple[str,str,str]) -> bool:
        """
        Check if the given fact, or (subject, relation, object) triple, is in the Knowledge Graph
        """
        source, relation, target = triple
        return target in self.edges.get(source, {}).get(relation, set())

## **Models and/or Methods**
We code up all 3 baselines provided in our task description and focus on potential improvements to each. Since no existing implementation of 



This one requires a little more in depth look + some code blocks. A super good way to do this is to just code a simple modl and test it on 1, 2, 3 hop hits@1 and it will perform pretty shit - the spec requires comoparisons to other models for full marks. Then we code the full model from paper 2 - it should be publicly avalible, we can copy and paste as long as we reference. We then make slight adjustments to it if we can think of any for some extra marks. These can be pretty minor, just to show we aren't straight up copy pasting. Then we train it with a sparse graph and see how it fairs. We probably also need something to create the reverse connections or loading triples or whatever this model requires - I didn't fully check. I'll show what it could look like more clearly (for any large code blocks, either include markdown or plenty of comments):

1. prep data (for dumb model)
2. write, train and test dumb model (dont need to display the results yet, but store)
3. prep data (for embedded model)
4. maybe a quick markdown of how this model actually works
5. code, train and test good model
6. tweak and retrain new model if we can
7. potential for halving data 

*also quick note, we need to save and clearly display the params we used like train size, epochs, batching, allat

In [ ]:
#We can put code here

## Results:
This one is super easy to drop marks and its not actually that much work. Just show data of shit, good and tweaked models. I assume data will just be hits@1 for 1, 2 and 3 hop. Try to use alot of graphs and tables to show how data improved. At this stage we can lightly explain the data and make some slight comaprisons to all the models we've done. The next section is discussion so I will see how much we actually need to do. Results and special formatting for jupyter can be very easily ai'ed to make it look alot better.


## Discussion
As I said, this one is pretty easy. Talk about your results, look at papers, really look at limitations and why they apear. Where possible offer alteratives to fix the probelsms or talk about differnt models fixing the problem but their other downsides i.e I am pretty sure one of the papers is more effective at 3 hop but it kind doesn't do well a sparce graph. 
